In [187]:
import random
import math

In [188]:
grid_size = 25
population_size = 6
generations = 15
mutation_rate = 0.1
alpha = 1000
beta = 2
gamma = 1

components_dimm = {"ALU":(5,5), "Cache":(7,4), "Control Unit":(4,4), "Register File":(6,6), "Decoder":(5,3), "Floating Unit":(5,5)}
component_names = ["ALU","Cache","Control Unit","Register File","Decoder","Floating Unit"]
connections = [("Register File","ALU"), ("Control Unit","ALU"), ("ALU","Cache"), ("Register File", "Floating Unit"), ("Cache", "Decoder"), ("Decoder", "Floating Unit")]

In [189]:
def get_center(chromosome):
    centers = {}
    for i in range(len(chromosome)):
        x, y = chromosome[i]
        w, h = components_dimm[component_names[i]]
        centers[component_names[i]] = (x + w/2, y + h/2)
    return centers

In [190]:
def fitness(chromosome):
    overlaps = 0
    for i in range(len(component_names)):
        for j in range(i+1, len(component_names)):
            a_left, a_bottom = chromosome[i]
            aw, ah = components_dimm[component_names[i]]
            b_left, b_bottom = chromosome[j]
            bw, bh = components_dimm[component_names[j]]
            a_right = a_left + aw
            a_top = a_bottom + ah
            b_right = b_left + bw
            b_top = b_bottom + bh

            if not (a_right <= b_left or a_left >= b_right or a_bottom >= b_top or a_top <= b_bottom):
                overlaps += 1

    centers = get_center(chromosome)
    total_distance = 0
    for start,end in connections:
        c1, c2 = centers[start], centers[end]
        total_distance += math.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)

    x_coords = []
    y_coords = []
    for i in range(len(chromosome)):
        x, y = chromosome[i]
        w, h = components_dimm[component_names[i]]
        x_coords.append(x)
        x_coords.append(x + w)
        y_coords.append(y)
        y_coords.append(y + h)
    area = (max(x_coords)-min(x_coords))*(max(y_coords)-min(y_coords))
    fitness_value = -(alpha * overlaps + beta * total_distance + gamma * area)
    return fitness_value, overlaps, total_distance, area

In [191]:
def crossover(p1, p2):
    point = random.randint(1, len(p1)-1)
    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]
    return c1, c2

In [192]:
def mutate(chromosome):
    if random.random() < mutation_rate:
        index = random.randint(0, len(chromosome)-1)
        coords = (random.randint(0, grid_size), random.randint(0, grid_size))
        chromosome[index] = coords
    return chromosome

In [193]:
def run_ga():
    population = []
    for i in range(population_size):
        chromosome = []
        for j in range(len(component_names)):
            coords = (random.randint(0,grid_size), random.randint(0,grid_size))
            chromosome.append(coords)
        population.append(chromosome)

    for gen in range(generations):
        population.sort(key=fitness, reverse=True)
        new_gen = population[:2]
        while len(new_gen) < population_size:
            p1, p2 = random.sample(population, 2)
            off1, off2 = crossover(p1, p2)
            new_gen.append(mutate(off1))
            if len(new_gen) < population_size:
                new_gen.append(mutate(off2))
        population = new_gen

    best = population[0]
    fitness_value, overlaps, total_distance, area = fitness(best)
    return best, fitness_value, overlaps, total_distance, area

In [194]:
best_chromosome, fitness_value, overlaps, total_distance, area = run_ga()
print(f"Total Fitness: {fitness_value:.2f}")
print(f"Wiring Length: {total_distance:.2f} | Bounding Area: {area:.2f} | Overlaps: {overlaps}")

Total Fitness: -659.09
Wiring Length: 73.05 | Bounding Area: 513.00 | Overlaps: 0
